# EN->NOB OPUS Fine-Tune (Colab)

This notebook installs dependencies and runs the full fine-tuning pipeline for `Helsinki-NLP/opus-mt-tc-big-en-gmq`.

Expected input:
- A zip named `finetune_bundle.zip` containing your `finetune/` folder (scripts + config).
- Dataset file(s): either
  - `generated_en_nob_military_train_ready.jsonl` (preferred), or
  - `generated_en_nob_military.jsonl` (raw, then cleanup scripts will run).


In [ ]:
# Install runtime deps (Colab/T4 friendly)
!pip -q install --upgrade pip
!pip -q install --upgrade --upgrade-strategy only-if-needed transformers datasets accelerate sentencepiece sacrebleu evaluate


In [ ]:
# Environment sanity check
import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))
else:
    print('No CUDA GPU detected. Training will be much slower on CPU.')


In [ ]:
# Create workspace
from pathlib import Path

WORK = Path('/content/translation_finetune_workspace')
WORK.mkdir(parents=True, exist_ok=True)
print('workspace:', WORK)


## Upload bundle
Upload one zip file from your machine (for example `finetune_bundle.zip`).

The zip should contain at minimum:
- `finetune/config/finetune_config.json`
- `finetune/scripts/fix_generated_dataset_ids.py`
- `finetune/scripts/build_train_ready_subset.py`
- `finetune/scripts/build_hf_dataset_splits.py`
- `finetune/scripts/train_opus_tc_big.py`

Optional but recommended inside the same zip:
- `finetune/data/generated_en_nob_military_train_ready.jsonl` (or raw `generated_en_nob_military.jsonl`)

If dataset is in the zip, you can skip the separate dataset upload cell.


In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if not zip_name:
    raise RuntimeError('Please upload a zip bundle containing the finetune folder')

zip_path = WORK / zip_name
zip_path.write_bytes(uploaded[zip_name])

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(WORK)

FINETUNE_DIR = WORK / 'finetune'
if not FINETUNE_DIR.exists():
    raise RuntimeError('Expected /content/translation_finetune_workspace/finetune after extraction')

print('finetune dir:', FINETUNE_DIR)

data_dir = FINETUNE_DIR / 'data'
train_ready = data_dir / 'generated_en_nob_military_train_ready.jsonl'
raw_generated = data_dir / 'generated_en_nob_military.jsonl'
print('train_ready_exists:', train_ready.exists())
print('raw_generated_exists:', raw_generated.exists())


## Optional dataset upload
Run this only if your zip did not include either:
- `generated_en_nob_military_train_ready.jsonl`, or
- `generated_en_nob_military.jsonl`


In [ ]:
from google.colab import files
from pathlib import Path

data_dir = FINETUNE_DIR / 'data'
data_dir.mkdir(parents=True, exist_ok=True)

train_ready = data_dir / 'generated_en_nob_military_train_ready.jsonl'
raw_generated = data_dir / 'generated_en_nob_military.jsonl'

if train_ready.exists() or raw_generated.exists():
    print('Dataset already present in zip, skipping upload')
else:
    data_uploaded = files.upload()
    for name, payload in data_uploaded.items():
        out_path = data_dir / name
        out_path.write_bytes(payload)
        print('saved:', out_path)

    if not train_ready.exists() and not raw_generated.exists():
        raise RuntimeError('Upload generated_en_nob_military_train_ready.jsonl or generated_en_nob_military.jsonl')


In [ ]:
# Normalize config paths for Colab workspace
import json

config_path = FINETUNE_DIR / 'config' / 'finetune_config.json'
cfg = json.loads(config_path.read_text(encoding='utf-8'))

cfg['paths']['raw_generated_jsonl'] = './data/generated_en_nob_military.jsonl'
cfg['paths']['fixed_ids_jsonl'] = './data/generated_en_nob_military_fixed_ids.jsonl'
cfg['paths']['malformed_lines_log'] = './results/generated_en_nob_military_malformed_lines.txt'
cfg['paths']['train_ready_jsonl'] = './data/generated_en_nob_military_train_ready.jsonl'
cfg['paths']['train_ready_report_json'] = './results/generated_en_nob_military_train_ready_report.json'
cfg['paths']['hf_train_jsonl'] = './data/opus_train.jsonl'
cfg['paths']['hf_eval_jsonl'] = './data/opus_eval.jsonl'
cfg['paths']['hf_test_jsonl'] = './data/opus_test.jsonl'
cfg['paths']['split_report_json'] = './results/opus_split_report.json'
cfg['paths']['training_output_dir'] = './outputs/opus_tc_big_en_nob_ft'
cfg['paths']['best_model_dir'] = './outputs/opus_tc_big_en_nob_ft_best'
cfg['paths']['training_metrics_json'] = './results/opus_tc_big_en_nob_ft_metrics.json'

config_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
print('updated config:', config_path)


In [ ]:
# T4 profile (higher GPU utilization, faster training)
import json

cfg_path = FINETUNE_DIR / 'config' / 'finetune_config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8'))

cfg['training']['fp16'] = True
cfg['training']['bf16'] = False
cfg['training']['per_device_train_batch_size'] = 12
cfg['training']['per_device_eval_batch_size'] = 12
cfg['training']['gradient_accumulation_steps'] = 1
cfg['training']['gradient_checkpointing'] = False
cfg['training']['dataloader_num_workers'] = 2
cfg['training']['num_beams_eval'] = 2
cfg['training']['logging_steps'] = 20
cfg['training']['eval_steps'] = 250
cfg['training']['save_steps'] = 250

cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding='utf-8')
print('updated T4 profile:', cfg_path)
print(json.dumps(cfg['training'], indent=2, ensure_ascii=False))
print('If you hit CUDA OOM, set per_device_train_batch_size/per_device_eval_batch_size to 8.')


In [ ]:
# Run cleanup (only needed if raw file exists and train_ready is missing)
import subprocess

raw_path = FINETUNE_DIR / 'data' / 'generated_en_nob_military.jsonl'
train_ready_path = FINETUNE_DIR / 'data' / 'generated_en_nob_military_train_ready.jsonl'

if train_ready_path.exists():
    print('train_ready file already present, skipping fix/cleanup scripts')
else:
    if not raw_path.exists():
        raise RuntimeError('No train_ready or raw dataset file found in finetune/data')
    print('running fix_generated_dataset_ids.py ...')
    subprocess.run(['python3', str(FINETUNE_DIR / 'scripts' / 'fix_generated_dataset_ids.py')], check=True, cwd=str(FINETUNE_DIR))
    print('running build_train_ready_subset.py ...')
    subprocess.run(['python3', str(FINETUNE_DIR / 'scripts' / 'build_train_ready_subset.py')], check=True, cwd=str(FINETUNE_DIR))


In [ ]:
# Build train/eval splits
import subprocess

subprocess.run(['python3', str(FINETUNE_DIR / 'scripts' / 'build_hf_dataset_splits.py')], check=True, cwd=str(FINETUNE_DIR))
print('split generation complete')


In [ ]:
# Start fine-tuning (live logs)
import subprocess

cmd = ['python3', '-u', str(FINETUNE_DIR / 'scripts' / 'train_opus_tc_big.py')]
proc = subprocess.Popen(
    cmd,
    cwd=str(FINETUNE_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='')

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'training failed with exit code {ret}')
print('training complete')


In [ ]:
# Show key outputs
from pathlib import Path
import json

metrics_path = FINETUNE_DIR / 'results' / 'opus_tc_big_en_nob_ft_metrics.json'
split_report = FINETUNE_DIR / 'results' / 'opus_split_report.json'

if split_report.exists():
    print('split report:')
    print(split_report.read_text(encoding='utf-8'))

if metrics_path.exists():
    print('
training metrics:')
    print(metrics_path.read_text(encoding='utf-8'))

print('
output dir:', FINETUNE_DIR / 'outputs' / 'opus_tc_big_en_nob_ft')
print('best model dir:', FINETUNE_DIR / 'outputs' / 'opus_tc_big_en_nob_ft_best')


In [ ]:
# Optional: zip best model and download
from pathlib import Path
import shutil
from google.colab import files

best_dir = FINETUNE_DIR / 'outputs' / 'opus_tc_big_en_nob_ft_best'
zip_base = FINETUNE_DIR / 'outputs' / 'opus_tc_big_en_nob_ft_best'

if not best_dir.exists():
    raise RuntimeError(f'Best model dir not found: {best_dir}')

archive_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(best_dir))
print('created:', archive_path)
files.download(archive_path)
